# **Preparing Dependancies**

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

import gc
import torch
from PIL import Image, ImageDraw
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionImg2ImgPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("device:", DEVICE)

# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
# CATATAN PENTING
# Rubrik menyebut model `runwayml/stable-diffusion-v1-5`. Namun RunwayML telah
# membubarkan organisasinya di Hugging Face, sehingga repo tersebut kini
# mengembalikan HTTP 404 dan tidak dapat dimuat sama sekali.
# Referensi: https://github.com/huggingface/diffusers/issues/9322
#
# Repo di bawah adalah MIRROR RESMI Stable Diffusion v1.5 dengan bobot dan
# arsitektur yang identik — bukan penggantian ke model lain.
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# Model dimuat SATU KALI dan dipakai ulang untuk seluruh task text-to-image
# (generate standar, hyperparameter, batch, dan pergantian scheduler).
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)
pipe.enable_attention_slicing()

print("model dimuat:", MODEL_ID)

## **Generate Image**

In [ ]:
# Negative prompt disalin persis sesuai ketentuan rubrik.
NEGATIVE_PROMPT = (
    "photorealistic, realistic, photograph, 3d render, messy, blurry, "
    "low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration"
)

SEED = 222

# Prompt yang SAMA dipakai pada kedua fungsi agar perbandingan objektif.
PROMPT = (
    "a cute cartoon astronaut floating in outer space, flat vector illustration, "
    "pastel color palette, clean lines, minimalist, children book style"
)


def generate_simple_image(prompt, negative_prompt, seed):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
    ).images[0]


img_simple = generate_simple_image(PROMPT, NEGATIVE_PROMPT, SEED)
display(img_simple)

## **Generate Image with Hyperparameter Configuration**

In [ ]:
def generate_advanced_image(prompt, negative_prompt, seed,
                            guidance_scale, num_inference_steps):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]


# Prompt, negative prompt, dan seed sengaja sama persis dengan sel sebelumnya.
img_advanced = generate_advanced_image(
    PROMPT, NEGATIVE_PROMPT, SEED,
    guidance_scale=7.5,
    num_inference_steps=50,
)
display(img_advanced)

## **Guidance Scale Comparison**

In [ ]:
def tampilkan_baris(gambar, judul, lebar=320):
    kecil = [g.resize((lebar, lebar)) for g in gambar]
    kanvas = Image.new("RGB", (lebar * len(kecil), lebar), "white")
    for i, g in enumerate(kecil):
        kanvas.paste(g, (i * lebar, 0))
    print(" | ".join(judul))
    return kanvas


CFG_VALUES = [1.5, 7.5, 15.0]
hasil_cfg = [
    generate_advanced_image(PROMPT, NEGATIVE_PROMPT, SEED,
                            guidance_scale=cfg, num_inference_steps=30)
    for cfg in CFG_VALUES
]
display(tampilkan_baris(hasil_cfg, [f"CFG {c}" for c in CFG_VALUES]))

### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah:**
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, kesesuaian dengan prompt, dan variasi visual yang terlihat."*

*   **Gambar dengan "Scale" Tinggi:**
*"Jelaskan perbedaan yang terlihat dibandingkan guidance scale rendah, terutama pada detail gambar dan kedekatannya dengan prompt."*

## **Inference Steps Comparison**

In [ ]:
# Step rendah (rentang 5-15) dibandingkan step tinggi (rentang 30-50).
STEP_VALUES = [10, 40]
hasil_step = [
    generate_advanced_image(PROMPT, NEGATIVE_PROMPT, SEED,
                            guidance_scale=7.5, num_inference_steps=s)
    for s in STEP_VALUES
]
display(tampilkan_baris(hasil_step, [f"{s} steps" for s in STEP_VALUES]))

### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah:**
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, ketajaman, serta kemungkinan munculnya noise atau artefak."*
*   **Gambar dengan "Step" Tinggi:**
*"Jelaskan perbedaan yang terlihat dibandingkan step rendah, terutama pada detail gambar, kehalusan hasil, dan stabilitas visual."*

## **Batch Inference from One Prompt**

In [ ]:
def buat_grid(gambar, baris=2, kolom=2):
    w, h = gambar[0].size
    grid = Image.new("RGB", (kolom * w, baris * h), "white")
    for i, g in enumerate(gambar[: baris * kolom]):
        grid.paste(g, ((i % kolom) * w, (i // kolom) * h))
    return grid


generator = torch.Generator(device=DEVICE).manual_seed(SEED)
batch = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    generator=generator,
    guidance_scale=7.5,
    num_inference_steps=30,
    num_images_per_prompt=4,      # empat gambar sekaligus dari satu prompt
).images

display(buat_grid(batch, baris=2, kolom=2))

## **Load Scheduler**

In [ ]:
SCHEDULERS = {
    "Euler A": EulerAncestralDiscreteScheduler,
    "DPM++": DPMSolverMultistepScheduler,
    "DDIM": DDIMScheduler,
}


def load_scheduler(pipe, scheduler_name):
    # Scheduler dibangun dari config scheduler yang sedang aktif, sehingga bobot
    # UNet/VAE/text-encoder tetap berada di VRAM — model tidak dimuat ulang.
    if scheduler_name not in SCHEDULERS:
        raise ValueError(f"scheduler tidak dikenal: {scheduler_name}")
    pipe.scheduler = SCHEDULERS[scheduler_name].from_config(pipe.scheduler.config)
    return pipe


# Bukti model tidak dimuat ulang: id objek UNet tetap sama sebelum dan sesudah.
print("UNet id sebelum :", id(pipe.unet))

hasil_scheduler = {}
for nama in SCHEDULERS:
    load_scheduler(pipe, nama)
    hasil_scheduler[nama] = generate_advanced_image(
        PROMPT, NEGATIVE_PROMPT, SEED,
        guidance_scale=7.5, num_inference_steps=30,
    )

print("UNet id sesudah :", id(pipe.unet))
display(tampilkan_baris(list(hasil_scheduler.values()), list(hasil_scheduler.keys())))

### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DPM++ Scheduler":**
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DDIM Scheduler":**
*"Jelaskan karakteristik gambar yang dihasilkan."*

# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Inpainting**

### **Load Model Inpainting**

In [ ]:
# Checkpoint inpainting memakai instance TERPISAH karena arsitekturnya memang berbeda:
# UNet inpainting menerima 9 input channel (4 latent + 4 masked-latent + 1 mask),
# sedangkan UNet text-to-image hanya 4 channel. Jadi ini bukan pemuatan ulang model
# yang sama, melainkan checkpoint yang berbeda.
#
# Repo `runwayml/stable-diffusion-inpainting` juga sudah 404, sehingga dipakai mirror.
INPAINT_MODEL_CANDIDATES = [
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    "botp/stable-diffusion-v1-5-inpainting",
]

inpaint_pipe = None
for repo in INPAINT_MODEL_CANDIDATES:
    try:
        inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
            repo, torch_dtype=DTYPE, safety_checker=None,
        ).to(DEVICE)
        inpaint_pipe.enable_attention_slicing()
        INPAINT_MODEL_ID = repo
        print("checkpoint inpainting dimuat:", repo)
        break
    except Exception as e:
        print(f"gagal memuat {repo}: {type(e).__name__}")

assert inpaint_pipe is not None, "Tidak ada mirror inpainting yang berhasil dimuat."

### **Manual Masking**

In [ ]:
SEED_INPAINT = 9      # ketentuan rubrik untuk inpainting


def buat_mask_kotak(size, kotak):
    # Konvensi: PUTIH (255) = area yang diganti, HITAM (0) = area dipertahankan.
    mask = Image.new("L", size, 0)
    ImageDraw.Draw(mask).rectangle(kotak, fill=255)
    return mask


def pratinjau_mask(image, mask, alpha=0.5):
    # Overlay merah untuk membantu proses trial and error penentuan koordinat.
    overlay = Image.new("RGB", image.size, (255, 0, 0))
    return Image.composite(
        Image.blend(image.convert("RGB"), overlay, alpha),
        image.convert("RGB"),
        mask,
    )


# Koordinat berikut ditentukan secara hardcode lewat trial and error:
# jalankan pratinjau_mask(), amati posisinya, geser angkanya, ulangi.
KOTAK_SATELIT = (300, 60, 480, 220)

mask_manual = buat_mask_kotak(img_advanced.size, KOTAK_SATELIT)
display(pratinjau_mask(img_advanced, mask_manual))

### **Generate**

In [ ]:
def inpaint_engine(image, mask, prompt, negative_prompt="", seed=SEED_INPAINT,
                   guidance_scale=7.5, num_inference_steps=40):
    w, h = image.size
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    return inpaint_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        width=w,
        height=h,
    ).images[0]


PROMPT_SATELIT = (
    "a broken damaged satellite with cracked solar panels floating in space, "
    "flat vector illustration, same pastel color palette, clean lines"
)

img_inpaint = inpaint_engine(
    image=img_advanced,
    mask=mask_manual,
    prompt=PROMPT_SATELIT,
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED_INPAINT,
)
display(img_inpaint)

## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

In [ ]:
from transformers import pipeline as hf_pipeline

SEG_MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"
segmenter = hf_pipeline(
    "image-segmentation",
    model=SEG_MODEL_ID,
    device=0 if DEVICE == "cuda" else -1,
)
print("model segmentation dimuat:", SEG_MODEL_ID)

### **Masking with Segmentation Model**

In [ ]:
import numpy as np


def _mask_dari_segmen(image, segmen_terpilih):
    gabungan = np.zeros((image.size[1], image.size[0]), dtype=np.uint8)
    for s in segmen_terpilih:
        m = np.array(s["mask"].convert("L").resize(image.size))
        gabungan = np.maximum(gabungan, m)
    return Image.fromarray(gabungan)


def buat_mask_segmentasi(image, label_prioritas=("sky", "background", "wall", "ceiling")):
    # Label yang dikenali model segmentation berbeda-beda tergantung isi gambar,
    # sehingga pemilihannya dibuat adaptif:
    #   1) pakai label prioritas pertama yang benar-benar terdeteksi;
    #   2) bila tidak ada, ambil segmen dengan area terluas.
    # Dengan begitu selalu ada mask yang valid untuk tahap generate berikutnya.
    segmen = segmenter(image)
    if not segmen:
        raise RuntimeError("Model segmentation tidak mendeteksi objek apa pun.")

    tersedia = [s["label"] for s in segmen]
    print("label terdeteksi:", tersedia)

    for target in label_prioritas:
        cocok = [s for s in segmen if s["label"].lower() == target.lower()]
        if cocok:
            print(f"label dipakai: '{target}' (dari daftar prioritas)")
            return _mask_dari_segmen(image, cocok), target

    terluas = max(segmen, key=lambda s: np.count_nonzero(np.array(s["mask"].convert("L"))))
    print(f"tidak ada label prioritas yang cocok — memakai segmen terluas: '{terluas['label']}'")
    return _mask_dari_segmen(image, [terluas]), terluas["label"]


mask_auto, label_dipakai = buat_mask_segmentasi(img_advanced)
display(pratinjau_mask(img_advanced, mask_auto))

### **Generate**

In [ ]:
PROMPT_AUTOMASK = (
    "colorful nebula and distant stars in deep space, "
    "flat vector illustration, pastel color palette, clean lines"
)

# Penjaga: memastikan mask hasil segmentasi benar-benar terbentuk sebelum dipakai.
assert mask_auto is not None, "mask_auto kosong — jalankan ulang sel Masking di atas."
assert np.count_nonzero(np.array(mask_auto)) > 0, (
    f"Mask dari label '{label_dipakai}' seluruhnya hitam, tidak ada area yang diganti. "
    "Ganti label_prioritas pada sel Masking dengan salah satu label yang tercetak di sana."
)

img_automask = inpaint_engine(
    image=img_advanced,
    mask=mask_auto,
    prompt=PROMPT_AUTOMASK,
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED_INPAINT,
)
display(img_automask)

## **Outpainting**

### **Prepare the Canvas**

In [ ]:
def prepare_outpainting(image, direction, expand_px=192):
    # Memperluas kanvas ke SATU arah: "kiri", "kanan", "atas", atau "bawah".
    w, h = image.size
    arah = {
        "kiri":  ((w + expand_px, h), (expand_px, 0)),
        "kanan": ((w + expand_px, h), (0, 0)),
        "atas":  ((w, h + expand_px), (0, expand_px)),
        "bawah": ((w, h + expand_px), (0, 0)),
    }
    if direction not in arah:
        raise ValueError(f"arah tidak dikenal: {direction}. Pilihan: {list(arah)}")

    ukuran_baru, posisi = arah[direction]
    kanvas = Image.new("RGB", ukuran_baru, (255, 255, 255))
    kanvas.paste(image, posisi)

    # Seluruh kanvas putih (diisi model), lalu area gambar asli dihitamkan.
    mask = Image.new("L", ukuran_baru, 255)
    ImageDraw.Draw(mask).rectangle(
        [posisi[0], posisi[1], posisi[0] + w - 1, posisi[1] + h - 1], fill=0
    )
    return kanvas, mask


# Input outpainting adalah GAMBAR HASIL INPAINTING sebelumnya.
kanvas_out, mask_out = prepare_outpainting(img_inpaint, "kanan", expand_px=192)
display(pratinjau_mask(kanvas_out, mask_out))

### **Generate**

In [ ]:
PROMPT_OUTPAINT = (
    "continuation of outer space scene with distant stars and nebula, "
    "flat vector illustration, same pastel color palette, clean lines"
)

img_outpaint = inpaint_engine(
    image=kanvas_out,
    mask=mask_out,
    prompt=PROMPT_OUTPAINT,
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED_INPAINT,
)
display(img_outpaint)

## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

In [ ]:
def prepare_zoom_out(image, zoom_factor=1.4):
    # Gambar diletakkan di TENGAH kanvas yang lebih besar, sehingga area yang diisi
    # membentuk cincin ke segala arah — efeknya seperti kamera mundur.
    w, h = image.size
    kw, kh = int(w * zoom_factor), int(h * zoom_factor)
    kw -= kw % 8            # dimensi harus kelipatan 8 untuk VAE
    kh -= kh % 8

    kanvas = Image.new("RGB", (kw, kh), (255, 255, 255))
    kiri, atas = (kw - w) // 2, (kh - h) // 2
    kanvas.paste(image, (kiri, atas))

    mask = Image.new("L", (kw, kh), 255)
    ImageDraw.Draw(mask).rectangle([kiri, atas, kiri + w - 1, atas + h - 1], fill=0)
    return kanvas, mask


kanvas_zoom, mask_zoom = prepare_zoom_out(img_inpaint, zoom_factor=1.4)
display(pratinjau_mask(kanvas_zoom, mask_zoom))

### **Generate**

In [ ]:
def zoom_out_bertahap(image, prompt, tahap=2, zoom_factor=1.4, ukuran_maks=768):
    # Perluasan dilakukan berulang: hasil tahap sebelumnya menjadi input tahap berikutnya.
    hasil = [image]
    saat_ini = image
    for i in range(tahap):
        if max(saat_ini.size) * zoom_factor > ukuran_maks:
            print(f"tahap {i + 1} dilewati — melebihi batas {ukuran_maks}px")
            break
        kanvas, mask = prepare_zoom_out(saat_ini, zoom_factor)
        saat_ini = inpaint_engine(
            image=kanvas, mask=mask, prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT, seed=SEED_INPAINT,
        )
        hasil.append(saat_ini)
        print(f"tahap {i + 1}: {kanvas.size}")
    return hasil


tahapan_zoom = zoom_out_bertahap(img_inpaint, PROMPT_OUTPAINT, tahap=2)
display(tampilkan_baris(tahapan_zoom, [f"tahap {i}" for i in range(len(tahapan_zoom))]))

## **Base + Refiner Image Generation**

In [ ]:
# CATATAN PENTING
# Rubrik meminta `denoising_end=0.8` (Stage 1) dan `denoising_start=0.8` (Stage 2).
# Kedua parameter tersebut HANYA tersedia pada pipeline SDXL base+refiner, sedangkan
# tips submission secara eksplisit melarang penggunaan SDXL karena rawan OOM.
#
# Pada Stable Diffusion 1.5, porsi denoising yang tersisa dikendalikan lewat `strength`:
#     denoising_end   = 0.8  ->  base menjalankan 80% langkah
#     denoising_start = 0.8  ->  refiner melanjutkan 20% sisanya  ->  strength = 0.2
#
# Stage 1 tetap menghasilkan LATENT (output_type="latent") sesuai bunyi rubrik.

DENOISING_SPLIT = 0.8
TOTAL_STEPS = 50

# Refiner berbagi SELURUH komponen dengan pipeline base — tidak ada bobot baru
# yang dimuat ke VRAM.
refiner_pipe = StableDiffusionImg2ImgPipeline(**pipe.components)
refiner_pipe.enable_attention_slicing()
print("refiner berbagi UNet dengan base:", refiner_pipe.unet is pipe.unet)


def generate_two_stage(prompt, negative_prompt, seed,
                       denoising_split=DENOISING_SPLIT, total_steps=TOTAL_STEPS):
    # ---- Stage 1: Base menghasilkan latent pada 80% langkah ----
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    latent = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=7.5,
        num_inference_steps=int(total_steps * denoising_split),
        output_type="latent",
    ).images

    # Latent didekode agar dapat dioper ke pipeline Img2Img.
    with torch.no_grad():
        decoded = pipe.vae.decode(latent / pipe.vae.config.scaling_factor).sample
    gambar_stage1 = pipe.image_processor.postprocess(decoded, output_type="pil")[0]

    # ---- Stage 2: Refiner menuntaskan 20% sisa denoising ----
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    hasil = refiner_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=gambar_stage1,
        generator=generator,
        guidance_scale=7.5,
        strength=round(1.0 - denoising_split, 2),
        num_inference_steps=total_steps,
    ).images[0]

    return gambar_stage1, hasil


stage1, stage2 = generate_two_stage(PROMPT, NEGATIVE_PROMPT, SEED)
display(tampilkan_baris([stage1, stage2],
                        ["Stage 1 (base, 80%)", "Stage 2 (refiner, 20%)"]))

gc.collect()
torch.cuda.empty_cache()